##### Esplorazione del dataset + draft di modelli per il task di predizione della complessità

In [47]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"   

In [48]:
import pandas as pd
import numpy as np

In [49]:
# Codice preso da https://huggingface.co/datasets/mpapucci/impacts, adattato per caricare solo il dominio Wikipedia, con e senza le feature di profiling.

from datasets import load_dataset

# Load all domains (core columns only — fastest, recommended for most tasks)
#ds = load_dataset("mpapucci/impacts", "all")

# Load a specific domain (core columns only):
ds_wikipedia = load_dataset("mpapucci/impacts", "wikipedia")
# ds = load_dataset("mpapucci/impacts", "public_administration")

# Load with full linguistic profiling features (~300 columns):
# ds = load_dataset("mpapucci/impacts", "all_profiling")
ds_wikipedia_profiling = load_dataset("mpapucci/impacts", "wikipedia_profiling")
# ds = load_dataset("mpapucci/impacts", "public_administration_profiling")

## Dataset schema — IMPACTS ( solo core columns)


| Column | Type | Description |
|---|---|---|
| idx | int | Unique row identifier |
| original_sentence_idx  | int | Unique identifier for the original sentence (multiple rows share the same original) |
| original_text | string | The original complex sentence (Italian) |
| simplification | string | The machine-generated simplified sentence |
| original_base | float | Read-IT base score for the original sentence |
| original_lexical | float | Read-IT lexical score for the original sentence |
| original_syntax | float | Read-IT syntactic score for the original sentence |
| original_all | float | Read-IT overall readability score for the original sentence |
| simplification_base | float | Read-IT base score for the simplification |
| simplification_lexical | float | Read-IT lexical score for the simplification |
| simplification_syntax | float | Read-IT syntactic score for the simplification |
| simplification_all | float | Read-IT overall readability score for the simplification |


In [50]:
# In media ci sono 10 semplificazioni per frase originale, con un punteggio di leggibilità associato a ciascuna semplificazione. 
# Qui mostriamo le semplificazioni per la frase originale con ID 1, ordinate per leggibilità decrescente.

original_id = 1
pairs = [r for r in ds_wikipedia["train"] if r["original_sentence_idx"] == original_id]
pairs_sorted = sorted(pairs, key=lambda x: x["simplification_all"], reverse=True)

print("Original:", pairs_sorted[0]["original_text"])
for p in pairs_sorted:
    print(f"  Readability {p['simplification_all']:.3f}:", p["simplification"])

## Conclusione: essendo generate sinteticamente effettivamente fanno un po' cacare,
# non mi sembrano vere semplificazioni e alcune sono allucinate, ma ha comunque senso usarle per i task

Original: Il trattato fu firmato a Monteleone nel luglio del 1497, che vide risolversi la questione in questo modo:
  Readability 0.310: Il trattato fu firmato a Monteleone nel luglio del 1497, che vide risolversi la questione.
  Readability 0.170: Nel 1497, un trattato fu firmato a Monteleone per risolvere la questione.
  Readability 0.082: Questa questione è stata risolta nel luglio 1497 con un trattato firmato a Monteleone.
  Readability 0.067: La questione fu risolta in questo modo.
  Readability 0.066: Nel 1497, il trattato di Monteleone ha risolto la questione di...
  Readability 0.054: Questa questione è stata risolta nel luglio 1497 con un trattato firmato a Monteleone


In [51]:
## Primo approccio semplice per costruire train e test, teniamo solo una semplificazione e non tutte e 10 
# perchè a noi interessa solo poter distinguere tra originali e semplificazioni, non distinguere tra semplificazioni migliori o peggiori.

import numpy as np, pandas as pd
from datasets import load_dataset
from sklearn.model_selection import GroupShuffleSplit

DOMAIN  = "wikipedia_profiling"
N_ROWS  = 40000
SEED    = 42
TEST_SZ = 0.2

df = (load_dataset("mpapucci/impacts", DOMAIN, split="train")
        .shuffle(seed=SEED).select(range(N_ROWS)).to_pandas())

orig_feats = [c for c in df.columns if c.endswith("_original")]
simp_feats = [c for c in df.columns if c.endswith("_simplification")]

## Rinomina le colonne lasciando solo "text" e "score" per originali e semplificazioni, e aggiunge una colonna booleana "is_original"

def strip(cols, suf): return {c: c[:-len(suf)] for c in cols}

orig = (df[["original_sentence_idx","original_text","original_all"] + orig_feats]
        .drop_duplicates("original_sentence_idx")
        .rename(columns={"original_text":"text","original_all":"score", **strip(orig_feats,"_original")}))
orig["is_original"] = True

simp = (df[["original_sentence_idx","simplification","simplification_all"] + simp_feats]
        .rename(columns={"simplification":"text","simplification_all":"score", **strip(simp_feats,"_simplification")}))
simp["is_original"] = False

## Combina originali e semplificazioni, converte le feature in numeriche, rimuove righe con testo o punteggio mancanti o vuoti, 
# e divide in train e test mantenendo i gruppi di frasi originali separati.

data = pd.concat([orig, simp], ignore_index=True)
feat_cols = sorted(set(strip(orig_feats,"_original").values()) &
                   set(strip(simp_feats,"_simplification").values()))
data[feat_cols] = data[feat_cols].apply(pd.to_numeric, errors="coerce")
data["score"]   = data["score"].astype("float32")
data = data.dropna(subset=["text","score"])
data = data[data["text"].str.len() > 0]

tr, te = next(GroupShuffleSplit(1, test_size=TEST_SZ, random_state=SEED)
              .split(data, groups=data["original_sentence_idx"]))
train, test = data.iloc[tr].copy(), data.iloc[te].copy()

print("esempi:", len(data), "| train:", len(train), "| test:", len(test))
print(data["is_original"].value_counts())
print("NaN gruppi:", data["original_sentence_idx"].isna().sum(),
      "| overlap:", len(set(train["original_sentence_idx"]) & set(test["original_sentence_idx"])))

esempi: 73413 | train: 58751 | test: 14662
is_original
False    40000
True     33413
Name: count, dtype: int64
NaN gruppi: 0 | overlap: 0


In [52]:
## Approccio definitivo: considera un delta minimo di leggibilità tra originale e semplificazione (in modo tale che possa considerarsi una vera semplificazione), 
# e cap a K semplificazioni per originale (scelte a caso in modo tale che siano rappresentative di diverse semplificazioni)

import os
from functools import lru_cache

import numpy as np
import pandas as pd
from datasets import load_dataset
from sklearn.model_selection import GroupShuffleSplit

# ── Config ───────────────────────────────────────────────────────────────────
DOMAIN  = "wikipedia_profiling"
N_ORIG  = 40_000   # numero di frasi originali da campionare
SEED    = 42
OUT_DIR = "data"

# ── Load ─────────────────────────────────────────────────────────────────────
print("Loading full dataset…")
df = load_dataset("mpapucci/impacts", DOMAIN, split="train").to_pandas()

# Campiona N_ORIG original_sentence_idx unici, poi tiene tutte le loro righe
all_orig_ids  = df["original_sentence_idx"].unique()
rng           = np.random.default_rng(SEED)
selected_ids  = set(rng.choice(all_orig_ids, size=min(N_ORIG, len(all_orig_ids)), replace=False))
df            = df[df["original_sentence_idx"].isin(selected_ids)].copy()
print(f"Originals selezionati: {len(selected_ids)} | Righe totali: {len(df)}")

# Tiene solo coppie in cui la semplificazione è effettivamente più semplificata dell'originale (delta minimo di leggibilità)
DELTA = 0.05
df = df[df["original_all"] - df["simplification_all"] > DELTA].copy()
print(f"Dopo filtro DELTA={DELTA}: {len(df)} righe "
      f"({df['original_sentence_idx'].nunique()} originals con almeno 1 semplificazione valida)")

orig_feats = [c for c in df.columns if c.endswith("_original")]
simp_feats = [c for c in df.columns if c.endswith("_simplification")]

def _strip(cols, suf):
    return {c: c[: -len(suf)] for c in cols}

orig = (
    df[["original_sentence_idx", "original_text", "original_all"] + orig_feats]
    .drop_duplicates("original_sentence_idx")
    .rename(columns={"original_text": "text", "original_all": "score",
                     **_strip(orig_feats, "_original")})
)
orig["is_original"] = True
valid_ids = set(df["original_sentence_idx"])
orig = orig[orig["original_sentence_idx"].isin(valid_ids)]

simp = (
    df[["original_sentence_idx", "simplification", "simplification_all"] + simp_feats]
    .rename(columns={"simplification": "text", "simplification_all": "score",
                     **_strip(simp_feats, "_simplification")})
)
simp["is_original"] = False

# ── Cap: K semplificazioni per originale, scelte a caso in modo tale che siano rappresentative di diverse semplificazioni ──

K = 1
simp = simp.sample(frac=1, random_state=SEED).reset_index(drop=True)   # mescola tutto
simp["_rank"] = simp.groupby("original_sentence_idx").cumcount()       # ordine casuale entro gruppo
simp = simp[simp["_rank"] < K].drop(columns="_rank").reset_index(drop=True)
print(f"Semplificazioni dopo cap K={K}: {len(simp)} "
      f"(media {len(simp)/simp['original_sentence_idx'].nunique():.1f} per originale)")

print(f"Semplificazioni dopo cap K={K}: {len(simp)} "
      f"(media {len(simp)/len(selected_ids):.1f} per originale)")

data = pd.concat([orig, simp], ignore_index=True)

feat_cols = sorted(
    set(_strip(orig_feats, "_original").values()) &
    set(_strip(simp_feats, "_simplification").values())
)
data[feat_cols] = data[feat_cols].apply(pd.to_numeric, errors="coerce")
data["score"]   = data["score"].astype("float32")
data = data.dropna(subset=["text", "score"])
data = data[data["text"].str.len() > 0]

# ── Split 60 / 20 / 20 per gruppo ────────────────────────────────────────────

# Passo 1: separa test (20%) da train_val (80%)

tv_idx, te_idx = next(
    GroupShuffleSplit(1, test_size=0.2, random_state=SEED)
    .split(data, groups=data["original_sentence_idx"])
)
train_val, test = data.iloc[tv_idx].copy(), data.iloc[te_idx].copy()

# Passo 2: dal train_val, separa val (25% di train_val ≈ 20% del totale)

tr_idx, va_idx = next(
    GroupShuffleSplit(1, test_size=0.25, random_state=SEED)
    .split(train_val, groups=train_val["original_sentence_idx"])
)
train, val = train_val.iloc[tr_idx].copy(), train_val.iloc[va_idx].copy()

print(f"Totale: {len(data)} | Train: {len(train)} | Val: {len(val)} | Test: {len(test)}")
for name, split in [("train/val", (train, val)), ("train/test", (train, test)), ("val/test", (val, test))]:
    overlap = set(split[0]["original_sentence_idx"]) & set(split[1]["original_sentence_idx"])
    print(f"Overlap {name}: {len(overlap)} (deve essere 0)")

Loading full dataset…
Originals selezionati: 40000 | Righe totali: 286123
Dopo filtro DELTA=0.05: 243428 righe (37692 originals con almeno 1 semplificazione valida)
Semplificazioni dopo cap K=1: 37692 (media 1.0 per originale)
Semplificazioni dopo cap K=1: 37692 (media 0.9 per originale)
Totale: 75384 | Train: 45228 | Val: 15078 | Test: 15078
Overlap train/val: 0 (deve essere 0)
Overlap train/test: 0 (deve essere 0)
Overlap val/test: 0 (deve essere 0)


In [54]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from scipy.stats import spearmanr

# peso 3 agli originali (minority) per bilanciare il rapporto 1:3
#weights = np.where(train["is_original"], 3.0, 1.0)

model = make_pipeline(SimpleImputer(strategy="median"),
                      StandardScaler(),
                      Ridge(alpha=1.0))
model.fit(train[feat_cols], train["score"])  # , ridge__sample_weight=weights)
pred = model.predict(test[feat_cols])

print("Spearman:", spearmanr(pred, test["score"]).correlation)
print("MAE:",  float(np.mean(np.abs(pred - test["score"].values))))
print("RMSE:", float(np.sqrt(np.mean((pred - test["score"].values)**2))))

# diagnostica per fascia (opzionale)
t = test.copy()
t["bin"] = pd.cut(t["score"], [0,0.2,0.4,0.6,0.8,1.0])
t["se"]  = (pred - t["score"].values)**2
print(t.groupby("bin", observed=True).agg(n=("score","size"),
        rmse=("se", lambda e: e.mean()**0.5)))


Spearman: 0.7951886311857144
MAE: 0.12096051085575386
RMSE: 0.15271617843326807
               n      rmse
bin                       
(0.0, 0.2]  4342  0.141325
(0.2, 0.4]  4114  0.124564
(0.4, 0.6]  3116  0.138070
(0.6, 0.8]  2304  0.186617
(0.8, 1.0]  1202  0.226116


In [55]:
from sklearn.svm import LinearSVR

# peso 3 agli originali (minority) per bilanciare il rapporto 1:3
#weights = np.where(train["is_original"], 3.0, 1.0)

model = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler(),
    LinearSVR(C=1.0, epsilon=0.0,
              loss="squared_epsilon_insensitive", dual=False,
              max_iter=10000, random_state=SEED))

model.fit(train[feat_cols], train["score"])  # , linearsvr__sample_weight=weights)
pred = model.predict(test[feat_cols])
print("Spearman:", spearmanr(pred, test["score"]).correlation)
print("MAE:",  float(np.mean(np.abs(pred - test["score"].values))))
print("RMSE:", float(np.sqrt(np.mean((pred - test["score"].values)**2))))


Spearman: 0.7951881569021864
MAE: 0.12096492488356513
RMSE: 0.15271780293487694


In [56]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, classification_report

clf = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(),
                    LogisticRegression(max_iter=1000, class_weight="balanced"))

clf.fit(train[feat_cols], train["is_original"].astype(int))
proba = clf.predict_proba(test[feat_cols])[:, 1]
pred  = (proba > 0.5).astype(int)

y = test["is_original"].astype(int)
print("AUC:", roc_auc_score(y, proba))
print(classification_report(y, pred, target_names=["semplice","complesso"], digits=3))

AUC: 0.8271920003689177
              precision    recall  f1-score   support

    semplice      0.753     0.736     0.744      7539
   complesso      0.742     0.758     0.750      7539

    accuracy                          0.747     15078
   macro avg      0.747     0.747     0.747     15078
weighted avg      0.747     0.747     0.747     15078



In [57]:
import numpy as np
from sklearn.metrics import (roc_auc_score, f1_score, precision_score,
                             recall_score, accuracy_score)

def vista_binaria(scores_pred, is_original, thr=None, criterion="f1"):
    """
    Valuta punteggi continui (regressione o probabilità) come binario
    complesso=1 / semplice=0.

    thr=None  -> cerca la soglia ottima sui dati passati  (usare su VAL)
    thr dato  -> applica quella soglia                     (usare su TEST)
    criterion -> come scegliere la soglia: "f1" | "balanced_acc" | "youden"
    """
    y = np.asarray(is_original).astype(int)
    s = np.asarray(scores_pred, dtype=float)
    auc = roc_auc_score(y, s)

    if thr is None:
        cand = np.quantile(s, np.linspace(0.05, 0.95, 91))
        def obj(t):
            p = (s > t).astype(int)
            if criterion == "f1":
                return f1_score(y, p, zero_division=0)
            if criterion == "balanced_acc":
                return 0.5 * (recall_score(y, p, pos_label=1, zero_division=0) +
                              recall_score(y, p, pos_label=0, zero_division=0))
            if criterion == "youden":   # recall_complesso + specificità - 1
                return (recall_score(y, p, pos_label=1, zero_division=0) +
                        recall_score(y, p, pos_label=0, zero_division=0) - 1)
            raise ValueError(criterion)
        thr = max(cand, key=obj)

    p = (s > thr).astype(int)
    return {
        "auc": auc,
        "acc": accuracy_score(y, p),
        "prec_complesso": precision_score(y, p, pos_label=1, zero_division=0),
        "rec_complesso":  recall_score(y, p, pos_label=1, zero_division=0),
        "f1_complesso":   f1_score(y, p, pos_label=1, zero_division=0),
        "thr": thr,
    }

In [58]:
pred_val  = model.predict(val[feat_cols])
pred_test = model.predict(test[feat_cols])

thr = vista_binaria(pred_val,  val["is_original"])["thr"]       # tara su VAL
res = vista_binaria(pred_test, test["is_original"], thr=thr)    # valuta su TEST
print(res)

{'auc': 0.7956462271855098, 'acc': 0.6990317018172172, 'prec_complesso': 0.644821928385291, 'rec_complesso': 0.886191802626343, 'f1_complesso': 0.7464804469273743, 'thr': np.float64(0.2776088096134942)}


In [7]:
import gc, torch
try:
    del model, trainer, train_ds, test_ds
except NameError:
    pass
gc.collect()
torch.cuda.empty_cache()

In [8]:
# =========================================================
# SEZIONE 3 — Transformer (BERT) in regressione
#   riusa gli stessi train/test della Sezione 1
# =========================================================
import numpy as np
from datasets import Dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer, DataCollatorWithPadding,
                          set_seed)
from scipy.stats import spearmanr

set_seed(SEED)
MODEL = "indigo-ai/BERTino"     # veloce; per più qualità: "dbmdz/bert-base-italian-xxl-cased"
tok = AutoTokenizer.from_pretrained(MODEL)

def make_ds(d):
    ds = Dataset.from_pandas(
        d[["text","score"]].rename(columns={"score":"labels"}).reset_index(drop=True))
    return ds.map(lambda b: tok(b["text"], truncation=True, max_length=128), batched=True)

train_ds, test_ds = make_ds(train), make_ds(test)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL, num_labels=1, problem_type="regression")

def compute_metrics(p):
    preds = p.predictions[0] if isinstance(p.predictions, tuple) else p.predictions
    pred = preds.squeeze(-1).astype("float64")
    lab  = p.label_ids.astype("float64")
    return {"spearman": spearmanr(pred, lab).correlation,
            "mae":  float(np.mean(np.abs(pred - lab))),
            "rmse": float(np.sqrt(np.mean((pred - lab)**2)))}

args = TrainingArguments(
    output_dir="bert_reg",
    eval_strategy="epoch", save_strategy="no",
    per_device_train_batch_size=16, per_device_eval_batch_size=16,
    num_train_epochs=3, learning_rate=2e-5,
    fp16=True,
    eval_accumulation_steps=8,   # aggiunge solo questo
    logging_steps=50, report_to="none", seed=SEED)

trainer = Trainer(model=model, args=args,
    train_dataset=train_ds, eval_dataset=test_ds,
    processing_class=tok, data_collator=DataCollatorWithPadding(tok),
    compute_metrics=compute_metrics)

trainer.train()
print(trainer.evaluate())

Map:   0%|          | 0/58751 [00:00<?, ? examples/s]

Map:   0%|          | 0/14662 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: indigo-ai/BERTino
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.weight  | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Spearman,Mae,Rmse
1,0.015620,0.017634,0.860379,0.099814,0.132794
2,0.013916,0.016348,0.866559,0.095261,0.127859
3,0.009454,0.016061,0.866029,0.094316,0.126730


Training Loss,Validation Loss,Epoch,Spearman,Mae,Rmse
0.009454,0.016061,3,0.866029,0.094316,0.126730


{'eval_loss': 0.016060518100857735, 'eval_spearman': 0.8660291649904059, 'eval_mae': 0.09431616548437677, 'eval_rmse': 0.12673010031753115}
